### Performing LOO-PIT on Field-Level MCMCs with Fixed Initial Conditions

This code book performs LOO-PIT analysis on an MCMC chain provided by Hugo Simon and Arnaud Di Mattia (University of Paris-Saclay). 
The chains are Field-Level inference chains of galaxy density counts over a 32 x 32 x 32 voxel space.
These chains specifically have their initial conditions fixed and only samples over the cosmological parameters. 

First we load the necessary packages

In [ ]:
Pkg.instantiate()
Pkg.resolve()

In [ ]:
# Pkg.add("Turing")
# Pkg.add("Capse")
# Pkg.add("NPZ")
# Pkg.add(url="https://github.com/JuliaCosmologicalLikelihoods/PlanckLite.jl")
# Pkg.add("BenchmarkTools")
# Pkg.add("Plots")
# Pkg.add("Optim")
# Pkg.add("ForwardDiff")
# Pkg.add("LinearAlgebra")
# Pkg.add("Pathfinder")
# Pkg.add("MicroCanonicalHMC")
# Pkg.add("Transducers")
# Pkg.add("StatsPlots")
# Pkg.add("PairPlots")
# Pkg.add("LaTeXStrings")
# Pkg.add("MCMCChains")
# Pkg.add("PSIS")
# Pkg.add("ArviZ")
# Pkg.add("ArviZPythonPlots")
# Pkg.add("IrrationalConstants")
# Pkg.add("DimensionalData")
# Pkg.add("Statistics")

In [ ]:
# using Turing
# using Capse
using NPZ
# # using PlanckLite
# using BenchmarkTools
# using Plots
# using Optim
# using ForwardDiff
# using Pathfinder
# #using MicroCanonicalHMC
# #using Transducers
# using StatsPlots
# #using PairPlots
# #using LaTeXStrings
# #using CairoMakie
# using MCMCChains
using PSIS
# include("utils.jl");
using ArviZ
using ArviZPythonPlots
using LinearAlgebra
using IrrationalConstants
using DimensionalData
using Statistics
using Plots

### PSIS

We begin by running Pareto Smoothed Importance Sampling (PSIS) on the full voxels, and storing the results.

The chains provided are a single file, which is loaded below. 

In [ ]:
#f = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/16_mesh_ic/logdf_mesh.npz")
f = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/SelectionTestData/lpt_64_fNL0_self_ov1.0_L500_tubes/logdf_mesh.npz")
g = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/SelectionTestData/lpt_64_fNL0_miss1s_ov1.0_L500/logdf_mesh.npz")

For the process of PSIS, we need the logpdf_mesh field, which are the log probabilities of each voxel that the chain has determined by fitting the model to the full data set. That is, the logpdf_mesh represent $p(y | \theta)$

In [ ]:
f["logpdf_mesh"]

Each of the 32 x 32 x 32 voxels has 4 x 512 delta logdensities from the chain corresponding to that voxel.

Below, we create an array of vectors called logpdf_mesh to store the data from the file. Each of the 32 x 32 x 32 elements has a vector corresponding to a single voxel, and the vector holds the log probability values.

In [ ]:
logpdf_mesh_tubes = Array{Vector{Float64}}(undef, 64, 64, 64)
logpdf_mesh_miss = Array{Vector{Float64}}(undef, 64, 64, 64)

for i in 1:64, j in 1:64, k in 1:64
    logpdf_mesh_tubes[i,j,k] = []
    logpdf_mesh_miss[i,j,k] = []
end

for i in 1:64, j in 1:64, k in 1:64
    append!(logpdf_mesh_tubes[i,j,k], vec(f["logpdf_mesh"][:,:,i,j,k]))
    append!(logpdf_mesh_miss[i,j,k], vec(g["logpdf_mesh"][:,:,i,j,k]))
end

Thus, log_pdf is a data object representing the 32 x 32 x 32 voxels, with each element holding 2048 log probability values collected from the file.

In [ ]:
logpdf_mesh_tubes

In [ ]:
logpdf_mesh_tubes[3,16,4]

Next, we perform Pareto Smoothed Importance Sampling (PSIS) on each voxel's vector of log pdf probabilities.

In [ ]:
psis_result_tubes = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 64, 64, 64);
psis_result_miss = Array{PSISResult{Float64, Vector{Float64}, Int64, Int64, PSIS.GeneralizedPareto{Float64}}}(undef, 64, 64, 64);

for i in 1:64, j in 1:64, k in 1:64
    psis_result_tubes[i, j, k] = psis(-logpdf_mesh_tubes[i,j,k])
    psis_result_miss[i, j, k] = psis(-logpdf_mesh_miss[i,j,k])
end

The importance weights and the pareto shape values are stored in their own 32 x 32 x 32 arrays so that the PSIS weights and the pareto shape value can be recalled easily for each voxel if needed.

In [ ]:
psis_weights_tubes = Array{Vector{Float64}}(undef, 64, 64, 64)
psis_weights_miss = Array{Vector{Float64}}(undef, 64, 64, 64)

for i in 1:64, j in 1:64, k in 1:64
    psis_weights_tubes[i,j,k] = psis_result_tubes[i,j,k].weights
    psis_weights_miss[i,j,k] = psis_result_miss[i,j,k].weights
end

In [ ]:
pareto_shapes_tubes = []
pareto_shapes_miss = []

for i in 1:64, j in 1:64, k in 1:64
    append!(pareto_shapes_tubes, psis_result_tubes[i,j,k].pareto_shape)
    append!(pareto_shapes_miss, psis_result_miss[i,j,k].pareto_shape)
end

In [ ]:
psis_plot_tubes = Plots.scatter(pareto_shapes_tubes; marker =:+, markersize=6, legend = false, linewidth=2)
hline!([0.5,0.7], linestyle=:dot, color =:black)
hline!([1.0], linestyle=:solid, color=:black)

psis_plot_miss = Plots.scatter(pareto_shapes_miss; marker =:+, markersize=6, legend = false, linewidth=2)
hline!([0.5,0.7], linestyle=:dot, color =:black)
hline!([1.0], linestyle=:solid, color=:black)

Plots.plot(psis_plot_tubes, psis_plot_miss, layout = (1, 2))

The plot above is of the Pareto shape parameters from PSIS. Because the shape parameters are all under the threshhold of 0.5, this informs that the importance sampling is well-behaved and reliable for probability reweighting in the Leave One Out approach.

Now that we know the weights are reliable, we can proceed with the LOO-PIT analysis. The PSIS weights inform how to reweight $p(y | \theta)$ determined by the MCMC logpdf_mesh field, into $p(y_{-i} | \theta)$, the LOO probability density. 

First load the logcdf_mesh field from the MCMC chain file. Similar to logpdf_mesh, this field has contains 4 x 512 samples corresponding to the CDF of that particular voxel. It is stored below in an array with each voxel having an array of all CDF samples

In [ ]:
logcdf_mesh_tubes = Array{Vector{Float64}}(undef, 64, 64, 64)
logcdf_mesh_miss = Array{Vector{Float64}}(undef, 64, 64, 64)

for i in 1:64, j in 1:64, k in 1:64
    logcdf_mesh_tubes[i,j,k] = []
    logcdf_mesh_miss[i,j,k] = []
end

for i in 1:64, j in 1:64, k in 1:64
    append!(logcdf_mesh_tubes[i,j,k], vec(f["logcdf_mesh"][:,:,i,j,k]))
    append!(logcdf_mesh_miss[i,j,k], vec(g["logcdf_mesh"][:,:,i,j,k]))
end

In [ ]:
logcdf_mesh_tubes

The CDF values are then able to be systemically reweighted by the PSIS weights in order to obtain the LOO-PIT final values.

In [ ]:
loo_pit_tubes = Array{Float64}(undef, 64, 64, 64)
loo_pit_miss = Array{Float64}(undef, 64, 64, 64)

for i in 1:64
    for j in 1:64
        for k in 1:64
            loo_pit_tubes[i,j,k] = dot(psis_weights_tubes[i,j,k], exp.(logcdf_mesh_tubes[i,j,k]))
            loo_pit_miss[i,j,k] = dot(psis_weights_miss[i,j,k], exp.(logcdf_mesh_miss[i,j,k]))
        end
    end
end

Plotting the LOO-PIT values in a histogram, we see the approximately uniform distribution of values as expected from a MCMC sampling based on a strong model.

In [ ]:
loo_pit_histogram_tubes = Plots.histogram(vec(loo_pit_tubes), legend=false, bins=20, normalize=:pdf, 
                                            ylims=(0,1.2), xticks=0:0.2:1.2, yticks=0:0.2:1.2, title="Tubes", xaxis="LOO-PIT", yaxis="density")
hline!([1], color=:red, linestyle=:dash)

loo_pit_histogram_miss = Plots.histogram(vec(loo_pit_miss), legend=false, bins=20, normalize=:pdf, 
                                            ylims=(0,1.2), xticks=0:0.2:1.2, yticks=0:0.2:1.2, title="Miss", xaxis="LOO-PIT", yaxis="density")
hline!([1], color=:red, linestyle=:dash)

Plots.plot(loo_pit_histogram_tubes, loo_pit_histogram_miss, layout = (1, 2))

In [ ]:
loo_pit_zavg_tubes = Matrix{Float64}(undef, 64, 64)
loo_pit_zavg_miss = Matrix{Float64}(undef, 64, 64)

for i in 1:64
    for j in 1:64
        total_tubes = 0
        total_miss = 0
        for k in 1:64
            total_tubes += loo_pit_tubes[i,j,k]
            total_miss += loo_pit_miss[i,j,k]
        end
        loo_pit_zavg_tubes[i,j] = total_tubes / 64
        loo_pit_zavg_miss[i,j] = total_miss / 64
    end
end

In [ ]:
clims_loo_pit_heatmap = extrema(filter(isfinite, vcat(
    vec(loo_pit_zavg_tubes),
    vec(loo_pit_zavg_miss)
)))

In [ ]:
loo_pit_heatmap_tubes = heatmap(loo_pit_zavg_tubes, c=cgrad(:RdBu, rev=true), clims=clims_loo_pit_heatmap, colorbar=false, 
                                    title="LOO-PIT Map Tubes", xaxis="voxel x", yaxis="voxel y", xticks=0:8:64, yticks=0:8:64)

loo_pit_heatmap_miss = heatmap(loo_pit_zavg_miss, c=cgrad(:RdBu, rev=true), clims=clims_loo_pit_heatmap, colorbar=true, 
                                    title="LOO-PIT Map Miss", xaxis="voxel x", yaxis="voxel y", xticks=0:8:64, yticks=0:8:64)

Plots.plot(loo_pit_heatmap_tubes, loo_pit_heatmap_miss, layout = (1, 2))

In [ ]:
f = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/SelectionTestData/lpt_64_fNL0_self_ov1.0_L500_tubes/obs.npz")
g = npzread("/Users/ethansmith/Desktop/Waterloo/Masters/LOO-PIT/loop_it_in_the_field/data_hugo/SelectionTestData/lpt_64_fNL0_miss1s_ov1.0_L500/obs.npz")

In [ ]:
count_mesh_tubes = f["count_mesh"]
count_mesh_miss = g["count_mesh"]

In [ ]:
along_losz_tubes = Matrix{Float64}(undef, 64, 64)
along_losz_miss = Matrix{Float64}(undef, 64, 64)

for i in 1:64
    for j in 1:64
        total_tubes = 0
        total_miss = 0
        for k in 1:64
            total_tubes += count_mesh_tubes[i,j,k]
            total_miss += count_mesh_miss[i,j,k]
        end
        along_losz_tubes[i,j] = total_tubes / 64
        along_losz_miss[i,j] = total_miss / 64
    end
end

In [ ]:
clims_count_heatmap = extrema(filter(isfinite, vcat(
    vec(along_losz_tubes),
    vec(along_losz_miss)
)))

In [ ]:
heatmap_tubes = heatmap(along_losz_tubes, c=:viridis, clims=clims_count_heatmap, colorbar=false,
                            title="Observed Counts Map Tubes", xaxis="Voxel x", yaxis="Voxel y", xticks=0:8:64, yticks=0:8:64,
                            xlims=(0.5, size(along_losz_tubes, 2) + 0.5), ylims=(0.5, size(along_losz_tubes, 1) + 0.5), margin=0Plots.mm)

In [ ]:
heatmap_miss = heatmap(along_losz_miss, c=:viridis, clims=clims_count_heatmap, colorbar=false,
                            title="Observed Counts Map Miss", xaxis="Voxel x", yaxis="Voxel y", xticks=0:8:64, yticks=0:8:64,
                            xlims=(0.5, size(along_losz_miss, 2) + 0.5), ylims=(0.5, size(along_losz_miss, 1) + 0.5), margin=0Plots.mm)

In [ ]:
using Plots

# ─────────────────────────────────────────────
# Row 1: heatmaps — shared colour scale
# ─────────────────────────────────────────────
p11 = heatmap_tubes
p12 = heatmap_miss


# ─────────────────────────────────────────────
# Row 2: heatmaps — shared colour scale
# ─────────────────────────────────────────────

p21 = loo_pit_heatmap_tubes
p22 = loo_pit_heatmap_miss


# ─────────────────────────────────────────────
# Row 3: histograms — shared y-axis
# ─────────────────────────────────────────────

p31 = loo_pit_histogram_tubes
p32 = loo_pit_histogram_miss


# ─────────────────────────────────────────────
# Combine
# ─────────────────────────────────────────────

layout = @layout [
    a{0.45w} b{0.55w} 
    c{0.45w} d{0.55w} 
    e{0.45w} f{0.55w} 
].    

Plots.plot(
    p11, p12,
    p21, p22,
    p31, p32;
    layout=layout,
    size=(1400, 1200)
)

In [ ]:
stephist(vec(loo_pit_miss), legend=false, bins=20, normalize=:pdf, 
                                            ylims=(0,1.2), xticks=0:0.2:1.2, yticks=0:0.2:1.2, title="Miss", xaxis="LOO-PIT", yaxis="density")



In [ ]:
systematic_loo_pit_values = []
no_systematic_loo_pit_values = [] 


for i in 1:64
    for j in 1:64
        for k in 1:64
            if 15 <= i <= 18 && 15 <= j <= 18
                append!(systematic_loo_pit_values, loo_pit_miss[i,j,k])
            else
                append!(no_systematic_loo_pit_values, loo_pit_miss[i,j,k])
            end
        end
    end
end


In [ ]:
stephist(vec(loo_pit_miss), label="Full LOO-PIT Values", legend=true, bins=20, normalize=:pdf, 
                                            xticks=0:0.2:1.0, yticks=0:0.2:1.8, title="Miss", xaxis="LOO-PIT", yaxis="density")

stephist!(systematic_loo_pit_values, label="Systematic LOO-PIT Values", bins=20, normalize=:pdf)
stephist!(no_systematic_loo_pit_values, label="Non-Systematic LOO-PIT Values", bins=20, normalize=:pdf)



In [ ]:
systematic_loo_pit_values = []
no_systematic_loo_pit_values = [] 


for i in 1:64
    for j in 1:64
        for k in 1:64
            if 3 <= i <= 6 && 3 <= j <= 6
                append!(systematic_loo_pit_values, loo_pit_miss[i,j,k])
            else
                append!(no_systematic_loo_pit_values, loo_pit_miss[i,j,k])
            end
        end
    end
end

In [ ]:
stephist(vec(loo_pit_miss), label="Full LOO-PIT Values", legend=true, bins=20, normalize=:pdf, 
                                            xticks=0:0.2:1.0, yticks=0:0.2:1.8, title="MIss", xaxis="LOO-PIT", yaxis="density")

stephist!(vec(systematic_loo_pit_values), label="Systematic LOO-PIT Values", bins=20, normalize=:pdf)
stephist!(vec(no_systematic_loo_pit_values), label="Non-systematic LOO-PIT Values", bins=20, normalize=:pdf)